In [1]:
# === Build sequence indices from flat per-horizon tables (Option 1: early fusion) ===
# Inputs (per H): data/modeling/datasets/train_basin_daily_exogenous_h{H}.parquet
# Outputs (per H): data/modeling/datasets/seqindex_h{H}_L{L}.parquet
#                  data/modeling/datasets/metadata_seq_h{H}_L{L}.json

import pandas as pd
from pathlib import Path
import json

# --- knobs ---
HORIZONS = [1, 2, 3, 5, 7]
L = 30  # window length (days)

In [2]:
# === Resolve Project Root ===

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/qingfangliu/bhutan_climate_modeling


In [10]:
DATASETS_DIR = PROJECT_ROOT / "data" / "modeling" / "datasets"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
for H in HORIZONS:
    in_path = DATASETS_DIR / f"train_basin_daily_exogenous_h{H}.parquet"
    df = pd.read_parquet(in_path)

    # types & order
    if "date_local" in df.columns:
        df["date_local"] = pd.to_datetime(df["date_local"])
    df = df.sort_values(["basin_id", "date_local"]).reset_index(drop=True)

    # --- choose features for X (drop engineered lags/rollups/APIs and non-feature cols) ---
    drop_cols = {"basin_id", "date_local", "discharge_cms", f"y_h{H}", "qc_any"}
    feature_cols = [
        c for c in df.columns
        if c not in drop_cols
        and ("_sum_" not in c)
        and ("_mean_" not in c)
        and ("api_d" not in c)
        and (not c.startswith("q_lag_"))
        and (not c.startswith("q_mean_"))
        and (not c.startswith("q_std_"))
    ]

    # --- make index rows ---
    rows = []
    for bid, g in df.groupby("basin_id", sort=False):
        g = g.reset_index(drop=True)
        n = len(g)
        for i in range(L - 1, n):  # window ends at i (t), covers [i-L+1 .. i]
            w_start = i - L + 1
            # Minimal NA guard on inputs; remove this if you want literally all windows
            if g.loc[w_start:i, feature_cols].isna().any().any():
                continue
            t_end = g.loc[i, "date_local"]
            t_start = g.loc[w_start, "date_local"]
            t_target = t_end + pd.Timedelta(days=H)
            # y_h{H} already exists at row i; no need to fetch here for the index
            rows.append({
                "basin_id": bid,
                "t_start": t_start,
                "t_end": t_end,
                "t_target": t_target,
                "H": H,
                "L": L
            })

    seqindex = pd.DataFrame(rows)
    out_idx = DATASETS_DIR / f"seqindex_h{H}_L{L}.parquet"
    seqindex.to_parquet(out_idx, index=False)

    meta = {
        "H": H,
        "L": L,
        "feature_columns": feature_cols,
        "n_windows": int(len(seqindex)),
        "n_basins": int(seqindex["basin_id"].nunique()) if len(seqindex) else 0,
        "source_file": in_path.name,
        "notes": "Inputs exclude lag/rollup/API/q_* engineered features; raw daily + seasonality + statics only."
    }
    out_meta = DATASETS_DIR / f"metadata_seq_h{H}_L{L}.json"
    with open(out_meta, "w") as f:
        json.dump(meta, f, indent=2, default=str)

    print(f"[h={H}] windows: {len(seqindex)} → {out_idx}")
    print(f"[h={H}] metadata → {out_meta}")

[h=1] windows: 29050 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h1_L30.parquet
[h=1] metadata → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/metadata_seq_h1_L30.json
[h=2] windows: 29047 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h2_L30.parquet
[h=2] metadata → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/metadata_seq_h2_L30.json
[h=3] windows: 29044 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h3_L30.parquet
[h=3] metadata → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/metadata_seq_h3_L30.json
[h=5] windows: 29038 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h5_L30.parquet
[h=5] metadata → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/metadata_seq_h5_L30.json
[h=7] windows: 29032 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h7_L30.parquet
[h=

In [11]:
# === Add H=0 (nowcast) flat targets from saved base tables ===

H = 0

exog_path = DATASETS_DIR / "train_basin_daily_exogenous.parquet"
arx_path  = DATASETS_DIR / "train_basin_daily_arx.parquet"

train_exog = pd.read_parquet(exog_path)
te = pd.read_parquet(arx_path)

# ensure order
for df in (train_exog, te):
    df["date_local"] = pd.to_datetime(df["date_local"])
    df.sort_values(["basin_id", "date_local"], inplace=True)
    df.reset_index(drop=True, inplace=True)

# EXOG
exog_h0 = train_exog.copy()
exog_h0["y_h0"] = exog_h0.groupby("basin_id")["discharge_cms"].shift(0)  # same-day target
out_exog = DATASETS_DIR / "train_basin_daily_exogenous_h0.parquet"
exog_h0.to_parquet(out_exog, index=False)
print(f"[EXOG h=0] rows: {len(exog_h0)} → {out_exog}")

# ARX (optional, to mirror your other outputs)
arx_h0 = te.copy()
arx_h0["y_h0"] = arx_h0.groupby("basin_id")["discharge_cms"].shift(0)
out_arx = DATASETS_DIR / "train_basin_daily_arx_h0.parquet"
arx_h0.to_parquet(out_arx, index=False)
print(f"[ARX  h=0] rows: {len(arx_h0)} → {out_arx}")


[EXOG h=0] rows: 29185 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous_h0.parquet
[ARX  h=0] rows: 29011 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx_h0.parquet


In [12]:
# === Build sequence index for H=0 (Option 1: early fusion, exogenous) ===
import pandas as pd, json
from pathlib import Path

H = 0
L = 30
DATASETS_DIR = PROJECT_ROOT / "data" / "modeling" / "datasets"

in_path = DATASETS_DIR / "train_basin_daily_exogenous_h0.parquet"
df = pd.read_parquet(in_path)
df["date_local"] = pd.to_datetime(df["date_local"])
df = df.sort_values(["basin_id", "date_local"]).reset_index(drop=True)

# exclude engineered lags/rollups/APIs and non-feature cols
drop_cols = {"basin_id", "date_local", "discharge_cms", f"y_h{H}", "qc_any"}
feature_cols = [
    c for c in df.columns
    if c not in drop_cols
    and ("_sum_" not in c)
    and ("_mean_" not in c)
    and ("api_d" not in c)
    and (not c.startswith("q_lag_"))
    and (not c.startswith("q_mean_"))
    and (not c.startswith("q_std_"))
]

rows = []
for bid, g in df.groupby("basin_id", sort=False):
    g = g.reset_index(drop=True)
    n = len(g)
    for i in range(L - 1, n):  # window [i-L+1 .. i] predicts same-day (H=0)
        w_start = i - L + 1
        if g.loc[w_start:i, feature_cols].isna().any().any():
            continue
        rows.append({
            "basin_id": bid,
            "t_start": g.loc[w_start, "date_local"],
            "t_end":   g.loc[i, "date_local"],
            "t_target": g.loc[i, "date_local"],  # same day for H=0
            "H": H,
            "L": L,
        })

seqindex = pd.DataFrame(rows)
out_idx  = DATASETS_DIR / f"seqindex_h{H}_L{L}.parquet"
seqindex.to_parquet(out_idx, index=False)

meta = {
    "H": H,
    "L": L,
    "feature_columns": feature_cols,
    "n_windows": int(len(seqindex)),
    "n_basins": int(seqindex["basin_id"].nunique()) if len(seqindex) else 0,
    "source_file": in_path.name,
    "notes": "Exogenous sequences; drop engineered lag/rollup/API/q_*; statics + raw daily only.",
}
out_meta = DATASETS_DIR / f"metadata_seq_h{H}_L{L}.json"
with open(out_meta, "w") as f:
    json.dump(meta, f, indent=2, default=str)

print(f"[h=0] windows: {len(seqindex)} → {out_idx}")
print(f"[h=0] metadata → {out_meta}")


[h=0] windows: 29095 → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/seqindex_h0_L30.parquet
[h=0] metadata → /Users/qingfangliu/bhutan_climate_modeling/data/modeling/datasets/metadata_seq_h0_L30.json
